# Chapter 07 — Intro to Unsupervised Learning: Live Examples

**Session 3 | Chapter 7 | Duration: ~20 min**

We explore what 'structure in data' means — without any labels.

**Key principle:** Unsupervised learning finds patterns that humans didn't label — the data speaks for itself.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, make_blobs, make_moons, make_circles
from sklearn.cluster import KMeans

sns.set_theme(style='whitegrid')
np.random.seed(42)
print('Ready!')

## 1. What Labeled vs Unlabeled Data Looks Like

Let's take a dataset we know (Iris) and "forget" the labels.

In [ ]:
iris = load_iris()
X = iris.data[:, 2:4]  # petal length and width — the most informative pair
y = iris.target         # the true labels (we'll hide them)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Without labels
axes[0].scatter(X[:, 0], X[:, 1], color='steelblue', alpha=0.6, s=60, edgecolors='white')
axes[0].set_xlabel('Petal Length (cm)')
axes[0].set_ylabel('Petal Width (cm)')
axes[0].set_title('UNLABELED — what unsupervised learning sees', fontsize=12)

# With labels
colors = {0: '#e74c3c', 1: '#3498db', 2: '#2ecc71'}
for cls, name in enumerate(iris.target_names):
    mask = y == cls
    axes[1].scatter(X[mask, 0], X[mask, 1], color=colors[cls],
                    label=name, alpha=0.7, s=60, edgecolors='white')
axes[1].set_xlabel('Petal Length (cm)')
axes[1].set_ylabel('Petal Width (cm)')
axes[1].set_title('LABELED — supervised learning version', fontsize=12)
axes[1].legend()

plt.suptitle('Same data — same visual structure — different perspective', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('In unsupervised learning, we see only the LEFT plot.')
print('Can you see 2-3 natural groups without the colors?')

## 2. What "Structure" Looks Like

sklearn can generate synthetic datasets with known structure — perfect for learning.

In [ ]:
# Different types of naturally clustered data
datasets = [
    ('Blob clusters (spherical)', *make_blobs(n_samples=300, centers=4, cluster_std=0.7, random_state=42)[:2]),
    ('Moon shapes (non-spherical)', *make_moons(n_samples=300, noise=0.07, random_state=42)[:2]),
    ('Concentric circles',  *make_circles(n_samples=300, noise=0.05, factor=0.4, random_state=42)[:2]),
]

# Show WITHOUT labels first
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (title, X_d, y_d) in zip(axes, datasets):
    ax.scatter(X_d[:, 0], X_d[:, 1], color='steelblue', alpha=0.6, s=30, edgecolors='white')
    ax.set_title(f'{title}\n(unlabeled)', fontsize=11)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('Three Types of Clustered Data — Can You Spot the Groups?', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Now reveal the true structure
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmap = plt.cm.tab10

for ax, (title, X_d, y_d) in zip(axes, datasets):
    ax.scatter(X_d[:, 0], X_d[:, 1], c=y_d, cmap=cmap, alpha=0.7, s=40, edgecolors='white')
    ax.set_title(f'{title}\n(true structure revealed)', fontsize=11)

plt.suptitle('True Structure — What Clustering Algorithms Must Discover', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. A Preview of Clustering — K-Means in Action

Even without labels, K-Means can find the structure in blob data.

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.7, random_state=42)

# K-Means clustering (no labels used!)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_blobs)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Unlabeled
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], color='steelblue', alpha=0.5, s=30)
axes[0].set_title('1. Unlabeled data — what the algorithm sees', fontsize=11)

# K-Means output
axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=cluster_labels, cmap='tab10', alpha=0.7, s=30)
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                c='black', marker='X', s=200, zorder=10, label='Centroids')
axes[1].set_title('2. K-Means discovered clusters (k=4)', fontsize=11)
axes[1].legend()

# True labels
axes[2].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='tab10', alpha=0.7, s=30)
axes[2].set_title('3. True groups (hidden from the algorithm)', fontsize=11)

plt.suptitle('K-Means Clustering: Discovering Groups Without Labels', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('K-Means found 4 groups that match the true structure — without ever seeing the labels!')
print('→ This is unsupervised learning in action.')

## 4. What About High-Dimensional Data?

Real data often has many features. Let's explore the digits dataset (64 features = 8×8 pixel images).

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f'Digits dataset: {X_digits.shape[0]} samples, {X_digits.shape[1]} features (8×8 pixels)')
print('Classes: digits 0–9')

# Show some example images
fig, axes = plt.subplots(2, 10, figsize=(16, 3.5))
for i, (ax, digit, label) in enumerate(zip(axes.flat, X_digits[:20], y_digits[:20])):
    ax.imshow(digit.reshape(8, 8), cmap='gray_r')
    ax.set_title(str(label), fontsize=12)
    ax.axis('off')
plt.suptitle('Sample Digits — 64-dimensional data', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Can we see structure in 64 dimensions? Let's reduce to 2 with PCA
pca = PCA(n_components=2, random_state=42)
X_digits_2d = pca.fit_transform(StandardScaler().fit_transform(X_digits))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Without labels (unsupervised view)
axes[0].scatter(X_digits_2d[:, 0], X_digits_2d[:, 1],
                color='steelblue', alpha=0.3, s=15)
axes[0].set_title('64-dim Digits compressed to 2D\n(no labels — unsupervised view)', fontsize=11)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# With labels
scatter = axes[1].scatter(X_digits_2d[:, 0], X_digits_2d[:, 1],
                           c=y_digits, cmap='tab10', alpha=0.5, s=15)
plt.colorbar(scatter, ax=axes[1], label='Digit')
axes[1].set_title('Same plot with true labels revealed', fontsize=11)
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.suptitle('PCA: Visualizing 64D Digits in 2D', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f'PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}')
print('Even 2 dimensions reveal clear digit clusters!')
print('→ This is the power of dimensionality reduction.')

## Summary

We saw:
- How labeled vs unlabeled data looks different (but is the same!)
- Three types of structure: blobs, moons, circles
- K-Means finding groups without any labels
- PCA compressing 64D to 2D while preserving digit structure

**Next: Chapter 08** — We'll master clustering algorithms!  
**Chapter 09** — We'll master dimensionality reduction!